In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/movies/archive/tmdb_5000_movies.csv
/kaggle/input/movies/archive/tmdb_5000_credits.csv


In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [3]:
!pip install nltk

In [4]:
import os


In [5]:
os.listdir('/kaggle/input/')


['movies']

In [6]:
movies =  pd.read_csv('/kaggle/input/movies/archive/tmdb_5000_movies.csv')
credits = pd.read_csv('/kaggle/input/movies/archive/tmdb_5000_credits.csv')

In [7]:
movies.head(1)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [8]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [9]:
movies = movies.merge(credits,on = "title")

In [10]:
movies= movies[['movie_id','title','genres','keywords','cast','overview','crew']]

In [11]:
movies.isnull().sum()

movie_id    0
title       0
genres      0
keywords    0
cast        0
overview    3
crew        0
dtype: int64

In [12]:
movies.dropna(inplace = True)

In [13]:
movies.isnull().sum()

movie_id    0
title       0
genres      0
keywords    0
cast        0
overview    0
crew        0
dtype: int64

In [14]:
movies.duplicated().sum()

0

In [15]:
import ast
ast.literal_eval

<function ast.literal_eval(node_or_string)>

In [16]:
def convert (obj):
    L=[]
    for i in ast.literal_eval(obj):
      L.append(  i['name'])
    return L
    

In [17]:
 movies['genres'] = movies['genres'].apply(convert)

In [18]:
movies.head(1)

,movie_id,title,genres,keywords,cast,overview,crew
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","In the 22nd century, a paraplegic Marine is di...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [19]:
movies['keywords'] = movies['keywords'].apply(convert)

In [20]:
def convert3 (obj):
    L=[]
    counter = 0
    for i in ast.literal_eval(obj):
        if counter!=3:
            L.append(  i['name'])
            counter+=1
        else:
            break
    return L
    

In [21]:
movies['cast'] = movies['cast'].apply(convert3)

In [22]:
def fetch_director (obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i['job']=='Director':
            L.append(  i['name'])
            break
    return L
    

In [23]:
movies['crew'] = movies['crew'].apply(fetch_director)


In [24]:
movies.head(1)

,movie_id,title,genres,keywords,cast,overview,crew
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","In the 22nd century, a paraplegic Marine is di...",[James Cameron]


In [25]:
 movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [26]:
movies['genres'] = movies['genres'].apply(lambda x : [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x : [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x : [i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x : [i.replace(" ","") for i in x])

In [27]:
movies ['tags'] = movies['overview']+ movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [28]:
new_df = movies [['movie_id','title','tags']]

In [29]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [30]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

/tmp/ipykernel_13/1824047427.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [31]:
new_df.head(1)

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."


In [32]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

/tmp/ipykernel_13/1380776331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [33]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 5000,stop_words = 'english')

In [34]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [35]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0])

In [36]:
len(cv.get_feature_names_out())

5000

In [37]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [38]:
def stem(text):
    y =[]
    for i in text.split():
       y.append( ps.stem(i))
    return "".join(y)

In [39]:
new_df['tags'] = new_df['tags'].apply(stem)

/tmp/ipykernel_13/3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [40]:
new_df['tags']

0       inthe22ndcentury,aparaplegmarinisdispatchtothe...
1       captainbarbossa,longbelievtobedead,hacomebackt...
2       acrypticmessagfrombond’pastsendhimonatrailtoun...
3       followthedeathofdistrictattorneyharveydent,bat...
4       johncarterisawar-weary,formermilitaricaptainwh...
                              ...                        
4804    elmariachijustwanttoplayhiguitarandcarrionthef...
4805    anewlywcouple'honeymoonisupendbythearrivofthei...
4806    "signed,sealed,delivered"introducadedicquartet...
4807    whenambitinewyorkattorneysamissenttoshanghaion...
4808    eversincthesecondgradewhenhefirstsawherine.t.t...
Name: tags, Length: 4806, dtype: object

In [41]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 5000,stop_words = 'english')

In [42]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [43]:
vectors


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [44]:
cv.get_feature_names_out()

array(['00', '000', '10', ..., 'zebra', 'zombies', 'zooeydeschanel'],
      dtype=object)

In [45]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [46]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)


In [47]:
recommend('Batman Begins')

Batman v Superman: Dawn of Justice
Avatar
Pirates of the Caribbean: At World's End
Spectre
The Dark Knight Rises
